# Dirac-3S Max Clique Benchmarking

In [ ]:
import numpy as np
import os
import time
import qci_client as qc
from qci_client.utilities import *
from maxclique import read_dimacs_clq, build_clique_hamiltonian
from maxclique import clique_number_from_energy, extract_clique
from maxclique import greedy_clique_from_weights

client = qc.QciClient(api_token="<API_TOKEN>", url="https://api.qci-prod.com")


### Set up dirac-3S solver

In [8]:
def convert_hamiltonian_to_poly_format(linear_terms, quadratic_terms, tol=0.0):
    """
    Convert (c, Q) of H(x) = x^T Q x + c^T x into Dirac-3 polynomial format.
    """
    c = np.asarray(linear_terms, dtype=float).ravel()
    Q = np.asarray(quadratic_terms, dtype=float)
    n = c.shape[0]

    if Q.shape != (n, n):
        raise ValueError(f"quadratic_terms has shape {Q.shape}, expected ({n}, {n})")

    poly_indices = []
    poly_coefficients = []

    # Linear terms
    for i in range(n):
        if abs(c[i]) > tol:
            poly_indices.append([0, i + 1])
            poly_coefficients.append(float(c[i]))

    # Quadratic terms -- fold the lower triangle onto the upper triangle
    Q_upper = np.triu(Q) + np.tril(Q, -1).T
    for i in range(n):
        for j in range(i, n):
            val = Q_upper[i, j]
            if abs(val) > tol:
                poly_indices.append([i + 1, j + 1])
                poly_coefficients.append(float(val))

    return poly_indices, poly_coefficients

def solve_dirac3(num_var, indices, coefficients, relaxation_schedule=1, sum_constraint=100, num_samples=1, device='dirac-3S'):
    data = [{"idx":idx, "val":val} for idx, val in zip(indices, coefficients)]
    file_quadratic = {
        "file_name": "dirac_3_continuous_variable_example",
        "file_config": {
            "polynomial": {
                "num_variables": num_var,
                "min_degree": 1,
                "max_degree": 2,
                "data": data,
            }
        }
    }
    # upload to qci client
    file_response = client.upload_file(file=file_quadratic)
    job_body = client.build_job_body(
        job_type = 'sample-hamiltonian',
        job_name = 'planted_quadratic', # user-defined (optional)
        job_params = {
            'device_type': device,
            'relaxation_schedule':relaxation_schedule,
            'sum_constraint':sum_constraint,
            'num_samples':num_samples,
        },
        polynomial_file_id=file_response['file_id'],
    )

    job_response = client.process_job(job_body=job_body)
    get_job_metrics_response = client.get_job_metrics(job_id=job_response["job_info"]["job_id"])
    return job_response, get_job_metrics_response

### load instance file and solve

In [9]:
instance_dir = "Instances/"
instance_name = "keller4.clq"
instance_path = os.path.join(instance_dir, instance_name)
try:
    A, n, m = read_dimacs_clq(instance_path)
    print("file loaded sucessfully")
except FileNotFoundError:
    print(f"{instance_path} does not exist")

file loaded sucessfully


In [10]:
name = os.path.splitext(instance_name)[0]
print(f"{name}: n = {n} vertices, m = {m} edges (declared)")
print(f"A shape: {A.shape}")
print(f"edges in A: {int(A.sum()) // 2}")
print(f"symmetric: {np.array_equal(A, A.T)}")
print(f"self-loops: {int(np.trace(A))}")
print(f"density: {A.sum() / (n * (n - 1)):.5f}")

keller4: n = 171 vertices, m = 9435 edges (declared)
A shape: (171, 171)
edges in A: 9435
symmetric: True
self-loops: 0
density: 0.64912


In [11]:
c = np.zeros(n)
H = build_clique_hamiltonian(A)

sum_constraint = 1
num_samples = 100
relaxation_schedule = 1
device = 'dirac-3s'

In [12]:
poly_indices, poly_coefficients = convert_hamiltonian_to_poly_format(
                    linear_terms=c,
                    quadratic_terms=H,   # the util mutates its input
                )

dirac_start = time.time()
response, metrics = solve_dirac3(
                        num_var=n,
                        indices=poly_indices,
                        coefficients=poly_coefficients,
                        relaxation_schedule=relaxation_schedule,
                        sum_constraint=sum_constraint,
                        device=device,
                        num_samples=num_samples)
dirac_end = time.time()
print(f"Dirac-3 Run complete. Response received.")
metrics = metrics["job_metrics"]
time_ns = metrics["time_ns"]
device = time_ns["device"][f"{device}"] #f"{device}"
runtimes = device["samples"]["runtime"]
post = device["samples"]["postprocessing_time"]
pre = device["samples"].get("preprocessing_time", 0)
solutions = response["results"]["solutions"]
energies = response['results']['energies']

runtimes_sec = [t*1e-9 for t in runtimes]
post_sec = [t*1e-9 for t in post]

print(f"Dirac-3 all samples energies:{[round(e, 2) for e in energies]}")
print(f"Dirac-3 all samples runtimes:{[round(t, 2) for t in runtimes]}")

2026-10-08 14:00:39 - Dirac allocation balance = 0 s (unmetered)
2026-10-08 14:00:39 - Job submitted: job_id='6ac7da47358b23d0cb371f16'
2026-10-08 14:00:39 - RUNNING
2026-10-08 14:05:01 - COMPLETED
2026-10-08 14:05:03 - Dirac allocation balance = 0 s (unmetered)
Dirac-3 Run complete. Response received.
Dirac-3 all samples energies:[-0.89, -0.89, -0.88, -0.88, -0.88, -0.88, -0.88, -0.88, -0.88, -0.88, -0.88, -0.87, -0.87, -0.87, -0.87, -0.87, -0.87, -0.87, -0.87, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -0.86, -

In [13]:
best = min(zip(energies, solutions, runtimes_sec, post_sec), key=lambda x: x[0])
dirac_energy, best_solution, best_runtime, best_postprocessing_time = best
preprocessing_time = pre*1e-9
omega = clique_number_from_energy(dirac_energy, sum_constraint)
support, isclique = extract_clique(best_solution, A)

# Round every restart, not just the lowest-energy one.
cliques = [greedy_clique_from_weights(sol, A) for sol in solutions]
greedy = max(cliques, key=len)

In [14]:
print(f"number of samples:{num_samples}, relaxatrion schedule:{relaxation_schedule}")
print(f"best energy over {num_samples} samples:{dirac_energy:.6f}")
print(f"support size:{len(support)}, support is a clique:{isclique}")
print(f"clique vertices (1-indexed):{(greedy + 1).tolist()}")
print(f"{'energy':>16}{'omega_est':>12}{'clique':>9}{'time (s)':>12}")
print(f"{dirac_energy:>16.6f}{omega:>12.3f}{len(greedy):>9}{best_runtime:>12.3f}")

# Never report a set that is not actually a clique.
assert extract_clique(np.isin(np.arange(n), greedy).astype(float), A)[1], \
    "greedy repair returned a non-clique"

number of samples:100, relaxatrion schedule:1
best energy over 100 samples:-0.888889
support size:14, support is a clique:False
clique vertices (1-indexed):[3, 16, 28, 77, 105, 111, 119, 137, 159]
          energy   omega_est   clique    time (s)
       -0.888889       9.000        9       2.130
